# 🌕 LunarSight — Master Autonomous Training & Execution Pipeline

**Autonomous Multi-Agent Water Ice Detection & Rover Traverse Pathfinding for Chandrayaan-2 DFSAR Telemetry**

This master notebook automates the entire end-to-end pipeline on **Google Colab (GPU Recommended: T4 / V100 / A100)**:
1. **Environment Setup & Dependency Installation** (`requirements_colab.txt`)
2. **Google Drive Integration** (auto-persists model checkpoints & output artifacts)
3. **Agent 1 — Data Ingestion & Co-registration** (DFSAR L1/L2 radar + DEM polar grid)
4. **Agent 2 — Despeckling & Noise Suppression** (Lee filter & Noise2Void model)
5. **Agent 3 — Polarimetric Analysis** (m-chi decomposition, S-Band $CPR > 1.2$, $DOP < 0.13$)
6. **Agent 4 — Weakly-Supervised U-Net Model Training** (pseudo-label generation + Deep Learning segmentation training)
7. **Agent 5 — Rover Traverse Pathfinding** (Slope-constrained A* / D* Lite navigation)
8. **Full Orchestrated Execution & Visual Diagnostics**

---

## ⚙️ Step 1: Environment Setup & Drive Mounting

In [ ]:
# Mount Google Drive for persistent model checkpoint storage
import os, sys, torch
from google.colab import drive
drive.mount('/content/drive')

# Workspace directory setup
WORKSPACE_DIR = '/content/Lunar-Sight'
if not os.path.exists(WORKSPACE_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {WORKSPACE_DIR}

%cd {WORKSPACE_DIR}/Lunar-Sight

# Install specialized Colab requirements
!pip install -q -r requirements_colab.txt

print(f"\n✅ Environment initialized. PyTorch Version: {torch.__version__}")
print(f"⚡ GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (No GPU detected)'}")

## 📡 Step 2: Agent 1 — Data Ingestion & Co-Registration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from agent1_ingestion.agent import agent1_node
from shared.state import LunarSightState

# Create execution state
state = LunarSightState()

# Run Agent 1 Node
print("▶ Executing Agent 1: Data Ingestion & Topographic Synthesis...")
out1 = agent1_node(state)
print("✅ Agent 1 complete. Output keys:", list(out1.keys()))

## 🧹 Step 3: Agent 2 — Despeckling & Radar Noise Suppression

In [ ]:
from agent2_despeckling.agent import agent2_node

# Update state with Agent 1 results
state.update(out1)

print("▶ Executing Agent 2: Despeckling Filter & Noise Suppression...")
out2 = agent2_node(state)
print("✅ Agent 2 complete. Output keys:", list(out2.keys()))

## 🔬 Step 4: Agent 3 — Polarimetric Decomposition & CPR Feature Maps

In [ ]:
from agent3_polarimetry.agent import agent3_node

state.update(out2)

print("▶ Executing Agent 3: Polarimetric Feature Extraction (m-chi, CPR, DOP)...")
out3 = agent3_node(state)
print("✅ Agent 3 complete. Output keys:", list(out3.keys()))

## 🧠 Step 5: Agent 4 — Weakly-Supervised U-Net Deep Learning Training

In [ ]:
# Set up persistent checkpoint directory on Google Drive
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/LunarSight/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

from agent4_segmentation.agent import agent4_node

state.update(out3)
state['checkpoint_dir'] = DRIVE_CHECKPOINT_DIR

print(f"▶ Executing Agent 4: U-Net Model Training (Saving checkpoints to {DRIVE_CHECKPOINT_DIR})...")
out4 = agent4_node(state)
print("✅ Agent 4 complete. Ice Mask Segmented successfully!")

## 🗺️ Step 6: Agent 5 — Slope-Constrained Rover Pathfinding

In [ ]:
from agent5_pathfinding.agent import agent5_node

state.update(out4)

print("▶ Executing Agent 5: Traverse Pathfinding & Safety Costmap Calculation...")
out5 = agent5_node(state)
print("✅ Agent 5 complete. Path planning finished!")

## 📊 Step 7: Visual Diagnostic Results & Telemetry Report

In [ ]:
state.update(out5)

# Plot End-to-End Mission Results
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("🌕 LunarSight — Mission Command Telemetry & Water Ice Detection", fontsize=16, fontweight='bold')

# 1. DEM / Slope
im0 = axes[0, 0].imshow(state.get('dem', np.zeros((100, 100))), cmap='terrain')
axes[0, 0].set_title('Polar DEM Topography')
plt.colorbar(im0, ax=axes[0, 0])

# 2. Despeckled Radar
im1 = axes[0, 1].imshow(state.get('despeckled_radar', np.zeros((100, 100))), cmap='gray')
axes[0, 1].set_title('Agent 2: Despeckled DFSAR')
plt.colorbar(im1, ax=axes[0, 1])

# 3. CPR Polarimetry Map
cpr_map = state.get('cpr_map', np.zeros((100, 100)))
im2 = axes[0, 2].imshow(cpr_map, cmap='magma', vmin=0, vmax=2.5)
axes[0, 2].set_title('Agent 3: S-Band CPR Map (Ice > 1.2)')
plt.colorbar(im2, ax=axes[0, 2])

# 4. Pseudo-labels
im3 = axes[1, 0].imshow(state.get('pseudo_labels', np.zeros((100, 100))), cmap='RdYlBu')
axes[1, 0].set_title('Agent 4: Weak Supervision Seeds')
plt.colorbar(im3, ax=axes[1, 0])

# 5. U-Net Ice Segmentation Mask
ice_mask = state.get('ice_mask', np.zeros((100, 100)))
im4 = axes[1, 1].imshow(ice_mask, cmap='Blues')
axes[1, 1].set_title(f'Agent 4: U-Net Ice Mask ({np.sum(ice_mask)} pixels)')
plt.colorbar(im4, ax=axes[1, 1])

# 6. Rover Traverse Route
path_map = state.get('costmap', np.zeros((100, 100))).copy()
axes[1, 2].imshow(path_map, cmap='gist_earth')
path = state.get('traverse_path', [])
if len(path) > 0:
    px, py = zip(*path)
    axes[1, 2].plot(py, px, color='#00D4AA', linewidth=2.5, label='Rover Path')
    axes[1, 2].legend(loc='upper right')
axes[1, 2].set_title('Agent 5: Rover Traverse Route')

plt.tight_layout()
plt.show()

print("\n🎉 MISSION EXECUTION COMPLETE!")
print(f"Checkpoints saved to: {DRIVE_CHECKPOINT_DIR}")